## Load Packages, Silently

In [ ]:
## Create a vector of the required packages for this analysis
req_packages <- c("ape", "Biostrings", "BSgenome",  "clevr", "cowplot", "GenomicFeatures", "ggplot2", "ggpubr", "ggtree", "ggvenn",
                  "gridExtra", "janitor", "nlme", "patchwork", "RColorBrewer", "stringr", 
                  "tidyverse", "treeio", "TreeTools", "vegan")

## load the packages, quietly
invisible(suppressWarnings(suppressMessages(
    lapply(req_packages, require, character.only = TRUE)
)))

## Identify variants in D. pseudoobscura mutant

### Set up Data

In [2]:
## load in vep results
raw_variants <- read_tsv("../workflow/VEP/variant_effect_output.txt", skip = 95) %>%
    janitor::clean_names()

## clean up the column information
all_variants <- raw_variants %>%
    separate_wider_delim(extra, ";", names = c("impact", "distance", "strand", "class", "symbol", "biotype"), 
                         too_few = "align_start", too_many = "drop") %>%
    mutate(biotype = case_when(grepl("BIOTYPE", symbol) ~ symbol,
                               TRUE ~ biotype),
           symbol = case_when(grepl("SYMBOL", class) ~ class,
                              TRUE ~ symbol),
           class = case_when(grepl("VARIANT_CLASS", distance) ~ distance,
                             grepl("VARIANT_CLASS", strand) ~ strand,
                             TRUE ~ class),
           strand = case_when(grepl("STRAND", distance) ~ distance,
                              TRUE ~ strand),
           distance = case_when(!grepl("DISTANCE", distance) ~ NA,
                                TRUE ~ distance)) %>%
    mutate(across(everything(), ~ str_replace(., ".*=", ""))) %>%
    separate_wider_delim(number_uploaded_variation, "_", names = c("chromosome", "position", "change"), too_few = "align_start", too_many = "drop") %>%
    mutate(position = as.integer(position))

## filter for only variants on the XR
variants <- all_variants %>%
    filter(grepl("ChrX", chromosome)) %>%
    filter(grepl("ChrX.1", chromosome) | grepl("ChrX.4", chromosome))

## visualize clean data
head(variants)

Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Rows: 5891958 Columns: 14
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (14): #Uploaded_variation, Location, Allele, Gene, Feature, Feature_type...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


### Check the summary information for variant effect

In [4]:
## count how many variants
nrow(variants)

[1] 306663

In [5]:
## count the number of SNPs
variants %>%
    filter(class == "SNV") %>%
    nrow()

[1] 240546

In [ ]:
## count the number of genes with mutations
variants %>%
    pull("symbol") %>%
    na.omit() %>%
    unique() %>%
    length()

In [10]:
## annotate mutation types
variant_type <- variants %>%
    separate_longer_delim(consequence, delim = ",") %>%
    mutate(type = case_when(grepl("UTR", consequence) ~ "UTR",
                            grepl("splice", consequence) ~ "splicing",
                            grepl("intron", consequence) ~ "intron",
                            grepl("frameshift", consequence) ~ "frameshift",
                            grepl("inframe", consequence) ~ "inframe",
                            grepl("start_lost", consequence) | grepl("stop_gained", consequence) ~ "nonsense",
                            grepl("missense", consequence) ~ "missense",
                            grepl("protein_altering", consequence) ~ "protein altering",
                            grepl("stream_gene_variant", consequence) ~ "distal"))

## count how many variants of each type
variant_type_number <- variant_type %>%
    select(type, symbol) %>%
    na.omit() %>%
    unique() %>%
    count(type)
variant_type_number

type,n
<chr>,<int>
UTR,545
distal,730
frameshift,42
inframe,135
intron,487
missense,369
nonsense,8
protein altering,4
splicing,303


In [ ]:
## define regulatory sequences
regulator <- c("upstream_gene_variant", "downstream_gene_variant", "3_prime_UTR_variant", "5_prime_UTR_variant")

## count the number of mutations per gene
multi_mutations <- variants %>%
    separate_longer_delim(consequence, ",") %>%
    filter(biotype == "protein_coding" & consequence != "synonymous_variant") %>%
    mutate(mutation_type = case_when(consequence %in% regulator ~ "Regulatory Element",
                             consequence == "intron_variant" ~ "Intron Variant",
                             grepl("splice", consequence) ~ "Splicing Variant",
                             grepl("inframe", consequence) ~ "Inframe InDel",
                             grepl("frameshift", consequence) ~ "Frameshift InDel",
                             consequence == "protein_altering_variant" ~ "Protein Altering",
                             consequence == "stop_gained" ~ "Nonsense Mutation",
                             consequence == "start_lost" ~ "Start Lost",
                             TRUE ~ "Other")) %>%
    group_by(mutation_type, symbol) %>%
    reframe(count = n())
head(multi_mutations)

In [23]:
## calculate the limit that 99% of genes are within
cap <- quantile(multi_mutations$count, probs = 0.75)
attributes(cap) <- NULL
cap

[1] 118

In [19]:
## find the highest number of mutations in a single gene
max(multi_mutations$count)

[1] 9615

In [20]:
## find the median number of mutations per gene
median(multi_mutations$count)

[1] 13

### Identify potential mutations of interest

In [22]:
## identify protein altering and nonsense mutations
potential_interest <- variants %>%
    filter(consequence == "protein_altering_variant" | consequence == "stop_gained") %>%
    select(symbol, position, change, consequence) %>%
    unique()
potential_interest

Rows: 16624 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): gene, max_organ, tau_tissue
dbl (2): tau, perc

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


symbol,position,change,consequence,max_organ,tau_tissue
<chr>,<int>,<chr>,<chr>,<chr>,<chr>
LOC6901779,645002,-/AGCCAC,protein_altering_variant,female_abdomen,ambiguous
LOC4815337,893839,G/C,stop_gained,female_gonad,ambiguous
LOC6902117,1625489,-/TGT,protein_altering_variant,female_gonad,ambiguous
LOC4815785,2706647,G/T,stop_gained,male_head,ambiguous
LOC4815774,2949241,C/T,stop_gained,male_digestive,ambiguous
LOC4815426,3468139,-/ACCAGT,protein_altering_variant,female_abdomen,ambiguous
LOC6902257,4221843,-/TGCTGC,protein_altering_variant,male_gonad,ambiguous
LOC4815659,4250261,C/A,stop_gained,female_thorax,ambiguous
LOC26532021,4393675,T/G,stop_gained,male_head,male_head


## Characterize gene expression in D. pseudoobscura mutant

### Set up Data

In [ ]:
## load in LogFC data
comparison_norm <- read_csv("../workflow/multigenome_rnaseq/results/logfc_normalized.csv")
head(comparison_norm)

In [ ]:
## load in results of Trinotate
ontology <- read_csv("../workflow/trinotate/dpse_goterms.csv") %>%
    select(-1)
head(ontology)

In [ ]:
## load in tau data
tau <- read.csv("../workflow/yang2018_rnaseq/results/tau_tissue_0.50.csv")
testes_genes <- tau %>%
    filter(tau_tissue == "male_gonad") %>%
    pull("gene")

In [ ]:
### create database for genes from GFF
dpse_ranges <- makeTxDbFromGFF("/home/labshare/data/genomes/obscura_group/pseudoobscura/GCF_009870125.1/genomic.gff", format = "gff")

### load in fasta of pseudoobscura
dpse_fasta <- readDNAStringSet("/home/labshare/data/genomes/obscura_group/pseudoobscura/GCF_009870125.1/GCF_009870125.1_UCI_Dpse_MV25_genomic.fna")

### Perform GO term analysis

In [ ]:
## get list of GO terms
go_terms <- unique(ontology$term)

## Look for GO terms enriched across clusters for each different group

## create empty dataframe to populate
go_compared <- data.frame(go_term = character(),
                          direction = character(),
                          go_genes = double(),
                          go_ratio = double(),
                          go_pvalue = double())

## get list of genes in cluster category
up_genes <- comparison_norm %>%
    filter(comparison == "sex ratio_mutant" & direction == "hypo") %>%
    pull("gene")
down_genes <- comparison_norm %>%
    filter(comparison == "sex ratio_mutant" & direction == "hyper") %>%
    pull("gene")

for (i in go_terms) {
    
    ## format dataframe based on mutation type and associated go term
    go_df <- ontology %>%
        mutate(signif = case_when(gene %in% up_genes ~ "signif",
                                  TRUE ~ "nontype"),
               go = case_when(term == i ~ "go",
                              TRUE ~ "nogo")) %>%
        group_by(signif, go) %>%
        count() %>%
        pivot_wider(names_from = go, values_from = n) %>%
        column_to_rownames("signif") %>%
        mutate_all(replace_na, replace = 0)

    ## get the number of genes with both mutation type and go term
    signif_go <- go_df["signif", "go"]

    ## only perform chi squared if there are more than 0 genes with go term
    if (signif_go > 0) {        
        
        ## perform chi-squared test
        go_chi <- go_df %>%
            as.matrix() %>%
            chisq.test()
        go_pvalue <- go_chi$p.value

        ## populate dataframe with data
        go_compared <- add_row(go_compared, go_term = i, 
                                            direction = "up",
                                            go_genes = signif_go, 
                                            go_ratio = signif_go/length(up_genes), 
                                            go_pvalue = go_pvalue)
        
    } 

    ## repeat for adaptive
    go_df <- ontology %>%
        mutate(signif = case_when(gene %in% down_genes ~ "signif",
                                  TRUE ~ "nontype"),
               go = case_when(term == i ~ "go",
                              TRUE ~ "nogo")) %>%
        group_by(signif, go) %>%
        count() %>%
        pivot_wider(names_from = go, values_from = n) %>%
        column_to_rownames("signif") %>%
        mutate_all(replace_na, replace = 0)
    signif_go <- go_df["signif", "go"]

    if (signif_go > 0) {        
        go_chi <- go_df %>%
            as.matrix() %>%
            chisq.test()
        go_pvalue <- go_chi$p.value
        go_compared <- add_row(go_compared, go_term = i, 
                                            direction = "down",
                                            go_genes = signif_go, 
                                            go_ratio = signif_go/length(down_genes), 
                                            go_pvalue = go_pvalue)
    } 

}

## perform multiple testing correction
go_compared$pvalue_corrected <- p.adjust(go_compared$go_pvalue, method = "BH")

In [ ]:
write.csv(go_compared, "intermediate/pseudoobscura_mutant_go.csv")

### Identify enrichment among differentially expressed genes

#### Identify testes specific enrichment

In [ ]:
## chi squared for all differentially expressed genes
chisq_testes <- comparison_norm %>%
    mutate(testes = case_when(gene %in% testes_genes ~ "testes",
                              TRUE ~ "other")) %>%
    filter(comparison == "sex ratio_mutant") %>%
    group_by(testes, significance) %>%
    count() %>%
    pivot_wider(names_from = significance, values_from = n) %>%
    column_to_rownames("testes") %>%
    as.matrix() %>%
    chisq.test()
chisq_testes$p.value

In [ ]:
## chi squared for different differentially expressed genes
chisq_up_testes <- comparison_norm %>%
    filter(comparison == "sex ratio_mutant") %>%
    mutate(testes = case_when(gene %in% testes_genes ~ "testes",
                              TRUE ~ "other"),
           increased = case_when(significance == "significant" & direction == "hypo" ~ "increased",
                                 TRUE ~ "other")) %>%
    group_by(testes, increased) %>%
    count() %>%
    pivot_wider(names_from = increased, values_from = n) %>%
    column_to_rownames("testes") %>%
    as.matrix() %>%
    chisq.test()
chisq_up_testes$p.value
chisq_down_testes <- comparison_norm %>%
    filter(comparison == "sex ratio_mutant") %>%
    mutate(testes = case_when(gene %in% testes_genes ~ "testes",
                              TRUE ~ "other"),
           increased = case_when(significance == "significant" & direction == "hyper" ~ "increased",
                                 TRUE ~ "other")) %>%
    group_by(testes, increased) %>%
    count() %>%
    pivot_wider(names_from = increased, values_from = n) %>%
    column_to_rownames("testes") %>%
    as.matrix() %>%
    chisq.test()
chisq_down_testes$p.value

In [ ]:
## chi squared for all differentially expressed genes with reference, as control
chisq_testes <- comparison_norm %>%
    mutate(testes = case_when(gene %in% testes_genes ~ "testes",
                              TRUE ~ "other")) %>%
    filter(comparison == "reference_mutant") %>%
    group_by(testes, significance) %>%
    count() %>%
    pivot_wider(names_from = significance, values_from = n) %>%
    column_to_rownames("testes") %>%
    as.matrix() %>%
    chisq.test()
chisq_testes$p.value
chisq_testes <- comparison_norm %>%
    mutate(testes = case_when(gene %in% testes_genes ~ "testes",
                              TRUE ~ "other")) %>%
    filter(comparison == "sex ratio_reference") %>%
    group_by(testes, significance) %>%
    count() %>%
    pivot_wider(names_from = significance, values_from = n) %>%
    column_to_rownames("testes") %>%
    as.matrix() %>%
    chisq.test()
chisq_testes$p.value

#### Identify chromosome specific enrichment

In [ ]:
## create GRanges object of all of the genes
genes <- genes(dpse_ranges)

## get list of differentially expresed genes
diff_genes <- comparison_norm %>%
    filter(significance == "significant") %>%
    pull("gene")

## get list of genes in each category 
upregulated_genes <- comparison_norm %>%
    filter(comparison == "sex ratio_mutant" & direction == "hypo" & significance == "significant") %>%
    pull("gene")
downregulated_genes <- comparison_norm %>%
    filter(comparison == "sex ratio_mutant" & direction == "hyper" & significance == "significant") %>%
    pull("gene")

## annotate all of the genes with location and expression
gene_characteristics <- genes %>%
    as_tibble() %>%
    select(seqnames, gene_id) %>%
    mutate(diff = case_when(gene_id %in% diff_genes ~ "diff",
                            TRUE ~ "nondiff"),
           dir = case_when(gene_id %in% upregulated_genes ~ "up",
                           gene_id %in% downregulated_genes ~ "down",
                           TRUE ~ "nondiff")) %>%
    rename(name = seqnames) %>%
    left_join(fasta_names, by = "name") %>%
    select(-c(original, name)) %>%
    mutate(x_chr = case_when(chromosome == "X" ~ "x_chr",
                            TRUE ~ "other_chr"))

## find the percentage of genes that are differentially expressed across chromosome
total_genes <- nrow(gene_characteristics)
signif_genes <- gene_characteristics %>%
    filter(diff == "diff") %>%
    nrow()

(signif_genes/total_genes)*100

In [ ]:
## get the percentage of genes on autosomes that are differentially expressed
nonx_genes <- gene_characteristics %>%
    filter(chromosome != "X") %>%
    nrow()
nonx_signif_genes <- gene_characteristics %>%
    filter(chromosome != "X" & diff == "diff") %>%
    nrow()
nonx_down_genes <- gene_characteristics %>%
    filter(chromosome != "X" & dir == "down") %>%
    nrow()

(nonx_signif_genes/nonx_genes)*100
(nonx_down_genes/nonx_genes)*100

In [ ]:
## get the percentage of genes on the X chromosome that are differentially expressed
x_genes <- gene_characteristics %>%
    filter(chromosome == "X") %>%
    nrow()
x_signif_genes <- gene_characteristics %>%
    filter(chromosome == "X" & diff == "diff") %>%
    nrow()
x_down_genes <- gene_characteristics %>%
    filter(chromosome == "X" & dir == "down") %>%
    nrow()

(x_signif_genes/x_genes)*100
(x_down_genes/x_genes)*100

In [ ]:
## calculate enrichment of differentially expressed genes on X chromosome
x_diff_chi <- gene_characteristics %>%
    select(gene_id, diff, x_chr) %>%
    group_by(diff, x_chr) %>%
    count() %>%
    pivot_wider(names_from = diff, values_from = n) %>%
    column_to_rownames("x_chr") %>%
    as.matrix() %>%
    chisq.test()
x_diff_chi$p.value

In [ ]:
## calculate enrichment of downregulated genes on X chromosome
up_diff_chi <- gene_characteristics %>%
    select(gene_id, dir, x_chr) %>%
    group_by(dir, x_chr) %>%
    filter(dir != "up") %>%
    count() %>%
    pivot_wider(names_from = dir, values_from = n) %>%
    column_to_rownames("x_chr") %>%
    as.matrix() %>%
    chisq.test()
up_diff_chi$p.value

In [ ]:
## calculate enrichment of upregulated genes on X chromosome
down_diff_chi <- gene_characteristics %>%
    select(gene_id, dir, x_chr) %>%
    group_by(dir, x_chr) %>%
    filter(dir != "down") %>%
    count() %>%
    pivot_wider(names_from = dir, values_from = n) %>%
    column_to_rownames("x_chr") %>%
    as.matrix() %>%
    chisq.test()
down_diff_chi$p.value

#### Identify intersection of testes specificity and X chromosome position

In [ ]:
## find if enriched in all differentially expressed genes
all_xtest <- gene_characteristics %>%
    mutate(xtest = case_when(x_chr == x_chr & gene_id %in% testes_genes ~ "yes",
                             TRUE ~ "no")) %>%
    group_by(diff,xtest) %>%
    count() %>%
    pivot_wider(names_from = diff, values_from = n) %>%
    column_to_rownames("xtest") %>%
    as.matrix() %>%
    chisq.test()
all_xtest$p.value

In [ ]:
## find if enriched in down genes
up_xtest <- gene_characteristics %>%
    filter(dir != "up") %>%
    mutate(xtest = case_when(x_chr == x_chr & gene_id %in% testes_genes ~ "yes",
                             TRUE ~ "no")) %>%
    group_by(dir, xtest) %>%
    count() %>%
    pivot_wider(names_from = dir, values_from = n) %>%
    column_to_rownames("xtest") %>%
    as.matrix() %>%
    chisq.test()
up_xtest$p.value

down_xtest <- gene_characteristics %>%
    filter(dir != "down") %>%
    mutate(xtest = case_when(x_chr == x_chr & gene_id %in% testes_genes ~ "yes",
                             TRUE ~ "no")) %>%
    group_by(dir ,xtest) %>%
    count() %>%
    pivot_wider(names_from = dir, values_from = n) %>%
    column_to_rownames("xtest") %>%
    as.matrix() %>%
    chisq.test()
down_xtest$p.value

## Associate Genes with parasperm content across obscure Group

### Set up Data

In [ ]:
## load in results of EvoGeneX
evogenex_results <- read_csv("../workflow/tools/EvoGeneX/evogenex_results.csv")

In [ ]:
## pull adaptive and constrained genes
adaptive_genes <- evogenex %>%
    filter(constraint == "constraint" & adaptive == "adaptive") %>%
    pull("gene")
constrained_genes <- evogenex %>%
    filter(constraint == "constraint") %>%
    pull("gene")
length(adaptive_genes)
length(constrained_genes)

In [ ]:
## load in results from CAGEE
credible_df <- read_csv("../workflow/CAGEE/credible_genes.csv")

In [ ]:
## get genes with credible changes at nodes with increases in gene expression (without conflicting if present in multiple)
credible_genes <- credible_df %>%
    pull("transcript_id")
length(unique(credible_genes))

In [ ]:
## load in gene expression data
raw_reads <- read_tsv("../workflow/tools/obscura_group_reads.tsv") %>%
    janitor::clean_names() %>%
    as.data.frame() %>%
    select(-desc)

## load in tree
tree <- ape::read.tree("../workflow/tools/select_obscura_tree.nwk")

### Cluster genes based on their phylogenetic independent contrasts across the obscura group

#### Filter for only the top 20% most variable genes

In [ ]:
## perform a PCA on the reads
pca <- raw_reads %>%
    column_to_rownames("gene_id") %>%
    as.matrix() %>%
    t() %>%
    prcomp()
summary(pca)

In [ ]:
## determines genes that contribute to variation
pca_loading <- pca$rotation %>%
    as.data.frame() %>%
    select(PC1, PC2, PC3) %>%
    rownames_to_column("gene") %>%
    pivot_longer(!gene, names_to = "pc", values_to = "loading") %>%
    group_by(pc) %>% 
    arrange(desc(abs(loading)))

## get top 20% of genes that explain variance
var_genes <- pca_loading %>%
    slice(1:(length(unique(pca_loading$gene))/5)) %>%
    pull("gene") %>%
    unique()

#### Calculate phylogenetic independent contrasts for each gene

In [ ]:
## reformat reads so each gene is a row and each species is a column
reads <- raw_reads %>%
    filter(gene_id %in% var_genes) %>% ## filter for only the most variable genes
    mutate_at(vars(-gene_id), as.double) %>%
    column_to_rownames("gene_id") %>%
    t() %>%
    as.data.frame() %>%
    rownames_to_column("species") %>%
    mutate(species = str_replace(species, "drosophila", "Drosophila")) %>%
    column_to_rownames("species")

## get list of species
species <- rownames(reads)

## create empty data frame
reads_pic <- species %>%
    as.data.frame() %>%
    rename(species = 1)

## populate dataframe with the phylogenetic independent contrasts
for (i in 1:ncol(reads)) {

    ## get name of gene
    gene_name <- names(reads)[[i]]

    ## create gene data frame
    gene_reads <- reads[,i]
    names(gene_reads) <- rownames(reads)

    ## calculate PIC
    gene_pic <- pic(gene_reads, tree)

    ## turn results into dataframe and rename
    gene_pic_df <- gene_pic %>%
        as.data.frame() %>%
        rownames_to_column("species")
    names(gene_pic_df)[2] <- gene_name
    # gene_pic_df$species <- species

    ## add gene to full PIC data frame
    reads_pic <- reads_pic %>%
        full_join(gene_pic_df, by = "species")
}

#### Cluster genes

In [ ]:
## create correlation matrix, only with non 0 genes
pic_trunc <- reads_pic %>%
    select(-c(1, 2))
reads_pic_mx <- reads_pic %>%
    t() %>%
    as.data.frame() %>%
    row_to_names(2) %>%
    mutate_all(as.double) %>%
    mutate(sums = colSums(pic_trunc)) %>%
    filter(sums != 0) %>%
    t()
pic_cor <- cor(reads_pic_mx)

## regularize correlations by threshold
ncol <- ncol(pic_cor)
nrow <- nrow(pic_cor)

pic_cor_reg <- pic_cor * (abs(pic_cor) > sqrt(log(ncol)/nrow))

## get euclidean distance between genes
pic_dist <- dist(pic_cor_reg, method = "euclidean") %>%
    na.omit()

In [ ]:
## cluster based on the difference between genes
pic_clust <- hclust(pic_dist)

## get the total number of genes 
gene_number <- length(pic_clust$height)/10

## test clustering method with 2 clusters
cut_2 <- cutree(pic_clust, 2)

## get within cluster sum of squares
wcss_2 <- sum((as.numeric(pic_clust$order) - cut_2)^2)

## get the number of genes in each cluster
cluster_numbers_2 <- cut_2 %>%
    as.data.frame() %>%
    rename(cluster = 1) %>%
    count(cluster, sort = TRUE) %>%
    filter(n != 1) %>%
    pull("n")

## get average and standard deviation
avg_number_2 <- mean(cluster_numbers_2)
stdev_2 <- sd(cluster_numbers_2)

## determine distance from optima
    ## distance from 0 WCSS and maximized average number of genes
dist_2 <- sqrt((wcss_2)^2 + (avg_number_2 - gene_number)^2)
    ## distance from 0 WCSS and 0 standard deviation and and maximized average number of genes
dist_stdev_2 <- sqrt((wcss_2)^2 + (stdev_2)^2 + (avg_number_2 - gene_number)^2)

## create data frame for 2 clusters analysis
k_df <- data.frame(k = 2, 
                   wcss = wcss_2,
                   avg_number = avg_number_2,
                   stdev = stdev_2,
                   dist = dist_2,
                   dist_stdev = dist_stdev_2)

## add cluster information for all additional clusters
for (k in 3:gene_number) {
    
    ## cut tree for cluster
    cut <- cutree(pic_clust, k)
    
    ## get within cluster sum of squares
    wcss <- sum((as.numeric(pic_clust$order) - cut)^2)

    ## get the number of genes in each cluster
    cluster_numbers <- cut %>%
        as.data.frame() %>%
        rename(cluster = 1) %>%
        count(cluster, sort = TRUE) %>%
        filter(n != 1) %>%
        pull("n")
    
    ## get average and standard deviation
    avg_number <- mean(cluster_numbers)
    stdev <- sd(cluster_numbers)

    ## determine distance from optima
        ## distance from 0 WCSS and maximized average number of genes
    dist <- sqrt((wcss)^2 + (avg_number - gene_number)^2)
        ## distance from 0 WCSS and 0 standard deviation and and maximized average number of genes
    dist_stdev <- sqrt((wcss)^2 + (stdev)^2 + (avg_number - gene_number)^2)

    ## add row to data frame
    k_df <- k_df %>%
        add_row(k = k, 
                wcss = wcss,
                avg_number = avg_number,
                stdev = stdev,
                dist = dist,
                dist_stdev = dist_stdev)

}

write.csv(k_df, "intermediate/cluster_number_df.csv")

In [ ]:
## find optimal number of k
k <- slice_min(k_df, dist_stdev, n = 1) %>%
    pull("k")
k

In [ ]:
## assign clusters based on optimal number
cluster_assignment <- cutree(pic_clust, k = k)

## clean up data frame
clusters <- cluster_assignment %>%
    as.data.frame() %>%
    rename(cluster = 1) %>%
    rownames_to_column("gene")

## scale data to more easily compare change
scale_change <- scale(reads) %>%
    t()

## add cluster assignments
pic_clusters <- scale_change %>%
    as.data.frame() %>%
    rownames_to_column("gene") %>%
    pivot_longer(!gene, names_to = "species", values_to = "scale_change") %>%
    left_join(parasperm, by = "species") %>%
    right_join(clusters, by = "gene") %>%
    right_join(cluster_count, by = "cluster") %>%
    mutate(name = paste0("Cluster ", cluster, "\n(", n, " genes)"))
write.csv(pic_clusters, "intermediate/pic_clusters.csv")

#### Calculate the heterogeneity of sperm associated genes in these clusters

In [ ]:
## prepare data by producing clustering and ground truths
### annotate genes
clusters_df <- clusters %>%
    mutate(credible = case_when(gene %in% credible_genes ~ 1,
                                TRUE ~ 2),
           adaptive = case_when(gene %in% adaptive_genes ~ 1,
                                TRUE ~ 2),
           parasperm = case_when(gene %in% credible_genes | gene %in% adaptive_genes ~ 1,
                                 TRUE ~ 2))

## get vectors
clusters_list <- clusters_df$cluster
credible_list <- clusters_df$credible
adaptive_list <- clusters_df$adaptive
parasperm_list <- clusters_df$parasperm

## how many credible and adaptive genes are clustered
genes <- clusters$gene
length(genes)
length(intersect(genes, credible_genes))
length(intersect(genes, adaptive_genes))

In [ ]:
## get the true v measure for credible and adaptive
credible_v <- clevr::v_measure(credible_list, clusters_list)
adaptive_v <- clevr::v_measure(adaptive_list, clusters_list)
parasperm_v <- clevr::v_measure(parasperm_list, clusters_list)

credible_v
adaptive_v
parasperm_v

In [ ]:
## create null model - randomly distribute credible and adaptive genes across k clusters
### set seed
set.seed(08282022)

### set randomization parameters
permutations <- 10000

## create empty vectors
permutations_credible <- vector()
permutations_adaptive <- vector()
permutations_parasperm <- vector()

## populate credible vector
for (i in 1:permutations) {

    ## randomly assign genes as credible
    permutation_list <- sample(credible_list)

    ## calculate test v measure
    permutations_credible[[i]] <- clevr::v_measure(permutation_list, clusters_list)

}

## populate adaptive vector
for (i in 1:permutations) {

    ## randomly assign genes as adaptive
    permutation_list <- sample(adaptive_list)

    ## calculate test v measure
    permutations_adaptive[[i]] <- clevr::v_measure(permutation_list, clusters_list)

}

## populate adaptive vector
for (i in 1:permutations) {

    ## randomly assign genes as adaptive
    permutation_list <- sample(parasperm_list)

    ## calculate test v measure
    permutations_parasperm[[i]] <- clevr::v_measure(permutation_list, clusters_list)

}

## p value is the number of permutations that have fewer pairs than the observed on average
credible_pvalue <- sum(permutations_credible >= credible_v, na.rm = TRUE)/permutations
adaptive_pvalue <- sum(permutations_adaptive >= adaptive_v, na.rm = TRUE)/permutations
parasperm_pvalue <- sum(permutations_parasperm >= adaptive_v, na.rm = TRUE)/permutations

credible_pvalue
adaptive_pvalue
parasperm_pvalue

In [ ]:
## store pvalues in data frame and save
pvalue_df <- data.frame(attr = c("credible", "adaptive", "parasperm"),
                        pvalue = c(credible_pvalue, adaptive_pvalue, parasperm_pvalue))

head(pvalue_df)

### Identify GO terms that are enriched for parasperm adaptive genes

In [ ]:
## look for enriched GO terms among the genes associated with increased parasperm
## get lists to iterate through
    ## genes
genes <- unique(evogenex$gene)
    ## ontology terms
go_terms <- unique(ontology$term)

## create empty dataframe to populate
go_compared <- data.frame(evolution = character(), 
                          go_term = character(),
                          go_genes = double(),
                          go_ratio = double(),
                          go_pvalue = double())

for (k in unique(evogenex$constraint)) {
    for (j in unique(evogenex$adaptive)) {

        evolution_genes <- evogenex %>%
            filter(constraint == k & adaptive == j) %>%
            pull("gene")

        for (i in go_terms) {    
                
            ## format dataframe based on mutation type and associated go term
            go_df <- ontology %>%
                mutate(evo = case_when(gene %in% evolution_genes ~ "evo",
                                            TRUE ~ "nontype"),
                       go = case_when(term == i ~ "go",
                                      TRUE ~ "nogo")) %>%
                group_by(evo, go) %>%
                count() %>%
                pivot_wider(names_from = go, values_from = n) %>%
                column_to_rownames("evo") %>%
                mutate_all(replace_na, replace = 0)

            ## get the number of genes with both mutation type and go term
            evo_go <- go_df["evo", "go"]

            ## only perform chi squared if there are more than 0 genes with go term
            if (evo_go > 0) {        
                
                ## perform chi-squared test
                go_chi <- go_df %>%
                    as.matrix() %>%
                    chisq.test()
                go_pvalue <- go_chi$p.value

                ## populate dataframe with data
                go_compared <- add_row(go_compared, 
                                            evolution = paste(k, j, sep = "_"),
                                            go_term = i, 
                                            go_genes = evo_go, 
                                            go_ratio = evo_go/length(evolution_genes), 
                                            go_pvalue = go_pvalue)
                
            }    
        }
    }
}

write.csv(go_compared, "intermediate/evogenex_go.csv")

### Identify GO terms that are enriched among branch associated genes

In [ ]:
## Look for GO terms enriched across clusters for each different group

## create empty dataframe to populate
go_compared <- data.frame(go_term = character(),
                          go_genes = double(),
                          go_ratio = double(),
                          go_pvalue = double())

for (i in go_terms) {
    
    ## format dataframe based on mutation type and associated go term
    go_df <- ontology %>%
        mutate(signif = case_when(gene %in% credible_genes ~ "signif",
                                  TRUE ~ "nontype"),
               go = case_when(term == i ~ "go",
                              TRUE ~ "nogo")) %>%
        group_by(signif, go) %>%
        count() %>%
        pivot_wider(names_from = go, values_from = n) %>%
        column_to_rownames("signif") %>%
        mutate_all(replace_na, replace = 0)

    ## get the number of genes with both mutation type and go term
    signif_go <- go_df["signif", "go"]

    ## only perform chi squared if there are more than 0 genes with go term
    if (signif_go > 0) {        
        
        ## perform chi-squared test
        go_chi <- go_df %>%
            as.matrix() %>%
            chisq.test()
        go_pvalue <- go_chi$p.value

        ## populate dataframe with data
        go_compared <- add_row(go_compared, go_term = i, 
                                            go_genes = signif_go, 
                                            go_ratio = signif_go/length(credible_genes), 
                                            go_pvalue = go_pvalue)
        
    } 
}

## perform multiple testing correction
go_compared$pvalue_corrected <- p.adjust(go_compared$go_pvalue, method = "BH")

write.csv(go_compared, "intermediate/credible_go.csv")

### Determine if there is enrichment for adaptive genes and branch associated genes on X or in testes specificity

In [ ]:
## are adaptive genes enriched on x chromosome
x_adaptive <- gene_characteristics %>%
    mutate(adaptive = case_when(gene_id %in% adaptive_genes ~ "adaptive",
                                TRUE ~ "non-adaptive")) %>%
    group_by(x_chr, adaptive) %>%
    count() %>%
    pivot_wider(names_from = x_chr, values_from = n) %>%
    column_to_rownames("adaptive") %>%
    as.matrix() %>%
    chisq.test()
x_adaptive$p.value

In [ ]:
## are adaptive genes enriched for testes specificity
testes_adaptive <- gene_characteristics %>%
    mutate(adaptive = case_when(gene_id %in% adaptive_genes ~ "adaptive",
                                TRUE ~ "non-adaptive"),
           testes = case_when(gene_id %in% testes_genes ~ "testes",
                              TRUE ~ "non-testes")) %>%
    group_by(testes, adaptive) %>%
    count() %>%
    pivot_wider(names_from = testes, values_from = n) %>%
    column_to_rownames("adaptive") %>%
    as.matrix() %>%
    chisq.test()
testes_adaptive$p.value

In [ ]:
## enrichment of adaptive in differentially expressed
diff_adaptive <- gene_characteristics %>%
    mutate(adaptive = case_when(gene_id %in% adaptive_genes ~ "adaptive",
                                TRUE ~ "non-adaptive")) %>%
    group_by(diff, adaptive) %>%
    count() %>%
    pivot_wider(names_from = diff, values_from = n) %>%
    column_to_rownames("adaptive") %>%
    as.matrix() %>%
    chisq.test()
diff_adaptive$p.value

In [ ]:
## find if adaptive enriched in directional genes
down_adaptive <- gene_characteristics %>%
    filter(dir != "up") %>%
    mutate(adaptive = case_when(gene_id %in% adaptive_genes ~ "adaptive",
                                TRUE ~ "non-adaptive")) %>%
    group_by(dir, adaptive) %>%
    count() %>%
    pivot_wider(names_from = dir, values_from = n) %>%
    column_to_rownames("adaptive") %>%
    as.matrix() %>%
    chisq.test()
down_adaptive$p.value

up_adaptive <- gene_characteristics %>%
    filter(dir != "down") %>%
    mutate(adaptive = case_when(gene_id %in% adaptive_genes ~ "adaptive",
                                TRUE ~ "non-adaptive")) %>%
    group_by(dir, adaptive) %>%
    count() %>%
    pivot_wider(names_from = dir, values_from = n) %>%
    column_to_rownames("adaptive") %>%
    as.matrix() %>%
    chisq.test()
up_adaptive$p.value

In [ ]:
## enrichment of constrained in differentially expressed
diff_constrained <- gene_characteristics %>%
    mutate(constrained = case_when(gene_id %in% constrained_genes ~ "constrained",
                                   TRUE ~ "non-constrained")) %>%
    group_by(diff, constrained) %>%
    count() %>%
    pivot_wider(names_from = diff, values_from = n) %>%
    column_to_rownames("constrained") %>%
    as.matrix() %>%
    chisq.test()
diff_constrained$p.value

down_constrained <- gene_characteristics %>%
    filter(dir != "up") %>%
    mutate(constrained = case_when(gene_id %in% constrained_genes ~ "constrained",
                                   TRUE ~ "non-constrained")) %>%
    group_by(dir, constrained) %>%
    count() %>%
    pivot_wider(names_from = dir, values_from = n) %>%
    column_to_rownames("constrained") %>%
    as.matrix() %>%
    chisq.test()
down_constrained$p.value

up_constrained <- gene_characteristics %>%
    filter(dir != "down") %>%
    mutate(constrained = case_when(gene_id %in% constrained_genes ~ "constrained",
                                   TRUE ~ "non-constrained")) %>%
    group_by(dir, constrained) %>%
    count() %>%
    pivot_wider(names_from = dir, values_from = n) %>%
    column_to_rownames("constrained") %>%
    as.matrix() %>%
    chisq.test()
up_constrained$p.value

## Compare parasperm associated genes with mutant information

### Identify enrichment of parasperm associated genes among differentially expressed genes

In [ ]:
## combine differential expression and parasperm data
annotated_logfc <- reads$gene_id %>%
    as.data.frame() %>%
    rename(gene = 1) %>%
    left_join(logfc, by = "gene") %>%
    mutate(logfc = -(logfc),
           credible = case_when(gene %in% credible_genes ~ "credible",
                                TRUE ~ "non-credible"),
           adaptive = case_when(gene %in% adaptive_genes ~ "adaptive",
                                TRUE ~ "non-adaptive"),
           significance = case_when(is.na(significance) ~ "not significant",
                                    TRUE ~ significance),
           color = case_when(is.na(color) ~ "not significant",
                             TRUE ~ color),
           direction = case_when(significance == "significant" & logfc < 0 ~ "hypo",
                                 significance == "significant" & logfc > 0 ~ "hyper",
                                 TRUE ~ "not significant"),
           testes_specificity = case_when(gene %in% testes_genes ~ "testes",
                                          TRUE ~ "other"),
           x = case_when(gene %in% x_genes ~ "x",
                         TRUE ~ "other")) %>%
    unique()

## compare number of significant genes to those with credible and adaptive
credible_logfc <- annotated_logfc %>%
    group_by(credible, significance) %>%
    count() %>%
    pivot_wider(names_from = significance, values_from = n) %>%
    column_to_rownames("credible") %>%
    mutate_all(replace_na, replace = 0) %>%
    as.matrix()
adaptive_logfc <- annotated_logfc %>%
    group_by(adaptive, significance) %>%
    count() %>%
    pivot_wider(names_from = significance, values_from = n) %>%
    column_to_rownames("adaptive") %>%
    mutate_all(replace_na, replace = 0) %>%
    as.matrix()

## perform chi squared test
chisq.test(credible_logfc)$p.value
chisq.test(adaptive_logfc)$p.value

In [ ]:
## now look at credible, separating increase and decrease
credible_up <- annotated_logfc %>%
    filter(direction != "hyper") %>%
    group_by(credible, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("credible") %>%
    mutate_all(replace_na, replace = 0) %>%
    as.matrix() %>%
    chisq.test()
credible_up$p.value

credible_down <- annotated_logfc %>%
    filter(direction != "hypo") %>%
    group_by(credible, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("credible") %>%
    mutate_all(replace_na, replace = 0) %>%
    as.matrix() %>%
    chisq.test()
credible_down$p.value

In [ ]:
## now look at adaptive, separating increase and decrease
adaptive_up <- annotated_logfc %>%
    filter(direction != "hypo") %>%
    group_by(adaptive, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("adaptive") %>%
    mutate_all(replace_na, replace = 0) %>%
    as.matrix() %>%
    chisq.test()
adaptive_up$p.value

adaptive_down <- annotated_logfc %>%
    filter(direction != "hyper") %>%
    group_by(adaptive, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("adaptive") %>%
    mutate_all(replace_na, replace = 0) %>%
    as.matrix() %>%
    chisq.test()
adaptive_down$p.value

### Identify GO terms associated with branch associated genes that are upregulated in the mutant

In [ ]:
## look for enriched GO terms in for credible, up-regulated genes

## get list of credible, up-regulated genes
upcred_genes <- annotated_logfc %>%
    filter(credible == "credible" & direction == "hypo") %>%
    pull("gene")

## create empty dataframe to populate
go_compared <- data.frame(go_term = character(),
                          go_genes = double(),
                          go_ratio = double(),
                          go_pvalue = double())

for (i in go_terms) {    
    
    ## format dataframe based on associated go term
    go_df <- ontology %>%
        mutate(signif = case_when(gene %in% upcred_genes ~ "signif",
                                    TRUE ~ "nontype"),
               go = case_when(term == i ~ "go",
                              TRUE ~ "nogo")) %>%
        group_by(signif, go) %>%
        count() %>%
        pivot_wider(names_from = go, values_from = n) %>%
        column_to_rownames("signif") %>%
        mutate_all(replace_na, replace = 0)

    ## get the number of genes with go term
    signif_go <- go_df["signif", "go"]

    ## only perform chi squared if there are more than 0 genes with go term
    if (signif_go > 0) {        
        
        ## perform chi-squared test
        go_chi <- go_df %>%
            as.matrix() %>%
            chisq.test()
        go_pvalue <- go_chi$p.value

        ## populate dataframe with data
        go_compared <- add_row(go_compared, go_term = i, 
                                            go_genes = signif_go, 
                                            go_ratio = signif_go/length(upcred_genes), 
                                            go_pvalue = go_pvalue)
        
    }    
}

## perform multiple testing correction
go_compared$pvalue_corrected <- p.adjust(go_compared$go_pvalue, method = "BH")

write.csv(go_compared, "intermediate/credible_up_go.csv")

## Investigate specific genes that would be of interest based on previous literature

### Set up Data

In [ ]:
## load in orthologs between D. melanogaster and D. pseudoobscura
mel_ortho <- read_delim("../workflow/trinotate/drosoma_pse_mel.txt", col_names = FALSE)
names(mel_ortho) <- c("dpse", "dmel", "paralog", "score")


### Search data for specific genes

In [ ]:
## nebenkern formation
## fzo (fuxxy onion) - not found (first gene related to mitochondria, cannot aggregate)
## Larp - not found (partitions mitochondria into nebenkern)
## Pink1 (LOC4815974) - not differential (maintains separation)
## Parkin - not found (maintains separation) -> tried park (uniprot), not differential
test <- mel_ortho %>%
    filter(dmel == "park") %>%
    pull("dpse")
annotated_logfc %>%
    filter(gene %in% test)

In [ ]:
## nebenkern unfurling
## Milton - not found (anchors nebernkern to nuclei and allows sliding of microtubules) -> tried milt (uniprot), not significant
## Lis-1 (LOC4803869) - not differential (anchors nebenkern to nuclei)
## Miro (LOC4801756) - not differential (allows sliding of cytosolic microtubules)
## Nebbish - not found (elongation is truncated) -> tried neb (uniprot), not significant
## Khc73 - not found (elongation is truncated) -> tried Khc-73 (uniprot), significant
## fascetto - not found (elongation is truncated)  -> tried feo (uniprot), significant
## nmd (LOC117183914) - not differential (decreased number of mitochondria)
## cnn (LOC4804379) - not differential (required for assembly of flagellar axoneme)
## Grip members - 3/7 differentially expressed (localize to basal body for microtubule formation)
test <- mel_ortho %>%
    filter(dmel == "feo") %>%
    pull("dpse")
annotated_logfc %>%
    filter(gene %in% test)

In [ ]:
## mitochondrial energetics
## Mic26-27 - not found (part of cristae organizing system)
## QIL1 (LOC4812916) - not differential (part of cristae organizing system)
## Mitofilin (LOC6896986) - not differential (part of cristae organizing system)
## Chchd3 (LOC4800334) - not differential (part of cristae organizing system)
## knon (LOC4803786) - not differential (electron transportation)
## knon (LOC4803786) - not differential (electron transportation)
## blw (LOC4803853) - differential (electron transportation)
test <- mel_ortho %>%
    filter(dmel == "Mic26-27") %>%
    pull("dpse")
annotated_logfc %>%
    filter(gene %in% test)

In [ ]:
## list of genes with differential expression
mito_genes <- c("Khc-73", "feo", "blw")
mel_ortho %>%
    filter(dmel %in% mito_genes | grepl("Grip", dmel)) %>%
    select(dpse, dmel) %>%
    rename(gene = dpse) %>%
    left_join(annotated_logfc, by = "gene") %>%
    select(-c(pvalue, direction, color))

In [ ]:
## acrosome genes
## Four way stop (fws) - not differential (golgi and endosomal trafficking regulators)
## Giotto (Gio) - not found (golgi and endosomal trafficking regulators)
## Rab11 - not differential (golgi and endosomal trafficking regulators)
## Nuclear fallout (nuf) - not differential (Rab11 effector Nuclear fallout)
## Brunelleschi (brun) - differential (golgi and endosomal trafficking regulators)
## Lectin Wheat Germ Agglutinin (WGA) - not found (secretory trafficking)
## Sneaky (snky) - differential (sperm plasma membrane breakdown during fertilization)
## Misfire (mfr) - differential (sperm plasma membrane breakdown)
test <- mel_ortho %>%
    filter(dmel == "mfr") %>%
    pull("dpse")
annotated_logfc %>%
    filter(gene %in% test)

In [ ]:
## golgi apparatus genes
## Golgins lava lamp (Lva) - differential
## GCC88 - not differential
## GM130 - not differential
## Golgi alpha-mannosidase II (GMII) - not found; (alpha-Man-IIa) - differential
## Golgi phosphoprotein-3 (GOLPHh3) - not found; (sau) - not differential
## Coatomer (COP I) - not found; (gammaCOP) - not significant
## Gga - not differential (clathrin adaptors)
## AP-1 - not found (clathrin adaptors)
test <- mel_ortho %>%
    filter(dmel == "AP-1") %>%
    pull("dpse")
annotated_logfc %>%
    filter(gene %in% test)

In [ ]:
## search for aquaporins
test <- mel_ortho %>%
    filter(grepl("AQP", dmel)) %>%
    pull("dpse")
annotated_logfc %>%
    filter(gene %in% test)

In [ ]:
## Odarant binding genes (Obp) - (odarant binding genes essential to male fertility)
test <- mel_ortho %>%
    filter(dmel == "lush") %>%
    pull("dpse")
annotated_logfc %>%
    filter(gene %in% test)
test <- mel_ortho %>%
    filter(grepl("Obp", dmel)) %>%
    rename(gene = dpse)
annotated_logfc %>%
    right_join(test, by = "gene")